# 02. Очистка данных

## Цель

Проверить пригодность последовательностей и аннотаций, согласовать правила формирования классов, обработать неоднозначные записи и точные повторы, подготовить очищенный набор данных для дальнейшей работы.

*В ноутбуке не выполняются обучение моделей, подбор признаков и исследование различий аминокислотного состава между классами.*

## Импорт библиотек

In [1]:
import pandas as pd
import re

import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.append(str(project_root))

from src.bioinformatic_functions import get_locations

## Загрузка tsv файла

In [2]:
df_raw = pd.read_csv("../data/raw/uniprot_human_swissprot_raw.tsv", sep="\t")

df = df_raw.copy()

print("Количество записей:", df.shape)
df.head()

Количество записей: (17342, 8)


,Entry,Entry Name,Protein names,Gene Names,Organism,Length,Sequence,Subcellular location [CC]
0,A0A087X1C5,CP2D7_HUMAN,Cytochrome P450 2D7 (EC 1.14.14.1),CYP2D7,Homo sapiens (Human),515,MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...,SUBCELLULAR LOCATION: Membrane {ECO:0000305}; ...
1,A0A096LP01,SIM26_HUMAN,Small integral membrane protein 26,SMIM26 LINC00493,Homo sapiens (Human),95,MYRNEFTAWYRRMSVVYGIGTWSVLGSLLYYSRTMAKSSVDQKDGS...,SUBCELLULAR LOCATION: Mitochondrion outer memb...
2,A0A0B4J2F0,PIOS1_HUMAN,Protein PIGBOS1 (PIGB opposite strand protein 1),PIGBOS1,Homo sapiens (Human),54,MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,SUBCELLULAR LOCATION: Mitochondrion outer memb...
3,A0A0C5B5G6,MOTSC_HUMAN,Mitochondrial-derived peptide MOTS-c (Mitochon...,MT-RNR1,Homo sapiens (Human),16,MRWQEMGYIFYPRKLR,SUBCELLULAR LOCATION: Secreted {ECO:0000269|Pu...
4,A0A0K2S4Q6,CD3CH_HUMAN,Protein CD300H (CD300 antigen-like family memb...,CD300H,Homo sapiens (Human),201,MTQRAGAAMLPSALLLLCVPGCLTVSGPSTVMGAVGESLSVQCRYE...,SUBCELLULAR LOCATION: [Isoform 1]: Membrane {E...


## Проверка аминокислотного алфавита

В первой версии оставляем последовательности только из 20 стандартных
аминокислот. Записи с другими символами сохраняем отдельно.

Ограничения длины пока не вводим.

In [3]:
mask = df["Sequence"].str.fullmatch(r"[ACDEFGHIKLMNPQRSTVWY]+", na=False)

df_other_symbols = df.loc[~mask].copy()

print("Последовательностей вне стандартного алфавита:", len(df_other_symbols))

Последовательностей вне стандартного алфавита: 23


In [4]:
df = df.loc[mask].copy()

print("Осталось записей:", len(df))

Осталось записей: 17319


## Записи с отдельными аннотациями изоформ

Некоторые записи описывают разные локализации для разных изоформ.

В первой версии такие записи откладываем, чтобы не назначить
последовательности локализацию другого варианта белка

In [5]:
mask = df["Subcellular location [CC]"].str.contains(r"\[\s*Isoform", case=False, regex=True, na=False)

df_isoforms = df.loc[mask].copy()

print(f"Записей с аннотациями изоформ:", len(df_isoforms))

df = df.loc[~mask].copy()

print("Осталось записей:", len(df))

Записей с аннотациями изоформ: 762
Осталось записей: 16557


## Выделение локализаций из текста

In [8]:
df["locations"] = df["Subcellular location [CC]"].apply(get_locations)

with pd.option_context("display.max_colwidth", None):
    display(
        df[["Entry", "Subcellular location [CC]", "locations"]].head(10)
    )

,Entry,Subcellular location [CC],locations
0,A0A087X1C5,SUBCELLULAR LOCATION: Membrane {ECO:0000305}; Multi-pass membrane protein {ECO:0000255}. Cytoplasm {ECO:0000305|PubMed:15051713}. Mitochondrion {ECO:0000269|PubMed:18838503}.,"[Membrane, Cytoplasm, Mitochondrion]"
1,A0A096LP01,"SUBCELLULAR LOCATION: Mitochondrion outer membrane {ECO:0000269|PubMed:33386847, ECO:0000269|PubMed:37009826}; Single-pass membrane protein {ECO:0000255}.",[Mitochondrion outer membrane]
2,A0A0B4J2F0,SUBCELLULAR LOCATION: Mitochondrion outer membrane {ECO:0000269|PubMed:31653868}; Single-pass membrane protein {ECO:0000255}.,[Mitochondrion outer membrane]
3,A0A0C5B5G6,"SUBCELLULAR LOCATION: Secreted {ECO:0000269|PubMed:25738459}. Mitochondrion {ECO:0000269|PubMed:29983246}. Nucleus {ECO:0000269|PubMed:29983246, ECO:0000269|PubMed:33473109}. Note=Translocates to the nucleus in response to metabolic stress in an AMPK-dependent manner. {ECO:0000269|PubMed:29983246}.","[Secreted, Mitochondrion, Nucleus]"
5,A0A0U1RRE5,"SUBCELLULAR LOCATION: Cytoplasm, P-body {ECO:0000269|PubMed:27918561}. Note=Localizes to P-bodies at low concentrations without dissociating them. {ECO:0000269|PubMed:27918561}.","[Cytoplasm, P-body]"
6,A0A1B0GTW7,SUBCELLULAR LOCATION: Membrane {ECO:0000255}; Single-pass type I membrane protein {ECO:0000255}.,[Membrane]
7,A0A2R8Y7D0,SUBCELLULAR LOCATION: Nucleus {ECO:0000269|PubMed:36369429}. Cell junction {ECO:0000269|PubMed:36369429}. Cytoplasm {ECO:0000250|UniProtKB:A0A1B0GRQ3}.,"[Nucleus, Cell junction, Cytoplasm]"
8,A0A8I5KQE6,"SUBCELLULAR LOCATION: Cell membrane {ECO:0000255|HAMAP-Rule:MF_03016}. Cytoplasm {ECO:0000255|HAMAP-Rule:MF_03016}. Nucleus {ECO:0000255|HAMAP-Rule:MF_03016}. Note=67LR is found at the surface of the plasma membrane, with its C-terminal laminin-binding domain accessible to extracellular ligands. 37LRP is found at the cell surface, in the cytoplasm and in the nucleus. Co-localizes with PPP1R16B in the cell membrane. {ECO:0000255|HAMAP-Rule:MF_03016}.","[Cell membrane, Cytoplasm, Nucleus]"
9,A0AV02,SUBCELLULAR LOCATION: Membrane {ECO:0000305}; Multi-pass membrane protein {ECO:0000305}.,[Membrane]
10,A0AV96,"SUBCELLULAR LOCATION: Nucleus {ECO:0000269|PubMed:24038582, ECO:0000269|PubMed:24916387}. Cytoplasm {ECO:0000269|PubMed:24916387}.","[Nucleus, Cytoplasm]"


## Соответствие локализаций нашим классам

Используем шесть предварительных классов:

- nucleus;
- cytoplasm;
- cell_membrane;
- mitochondrion;
- secreted;
- endoplasmic_reticulum.

In [7]:
location_classes = {
    "Nucleus": "nucleus",
    "Cytoplasm": "cytoplasm",
    "Cytosol": "cytoplasm",
    "Cell membrane": "cell_membrane",

    "Mitochondrion": "mitochondrion",
    "Mitochondrion matrix": "mitochondrion",
    "Mitochondrion inner membrane": "mitochondrion",
    "Mitochondrion outer membrane": "mitochondrion",
    "Mitochondrion intermembrane space": "mitochondrion",

    "Secreted": "secreted",

    "Endoplasmic reticulum": "endoplasmic_reticulum",
    "Endoplasmic reticulum membrane": "endoplasmic_reticulum",
    "Endoplasmic reticulum lumen": "endoplasmic_reticulum"
}

def get_class(location):
    main_location = location.split(",", 1)[0].strip()

    return location_classes.get(main_location)

## Назначение метки

Оставляем запись, если:

1. Все выделенные локализации распознаны.
2. Они относятся к одному нашему классу.
3. Хотя бы одно утверждение об этом классе содержит ECO:0000269.

Если указана неизвестная локализация или несколько разных классов,
запись пока откладываем.